# Etapa 1 — Diagnóstico y preparación de los datos

Encuesta de Experiencia Comercial a Distribuidoras (CRECE CON VALES).

Objetivo de este notebook: entender la estructura del archivo fuente, calidad de los registros, y dejar una base limpia para las siguientes etapas (EDA / clustering / NLP).

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RUTA_EXCEL = "../data/raw/Encuesta de Experiencia Comercial DV_concentrado.xlsx"
xls = pd.ExcelFile(RUTA_EXCEL)
xls.sheet_names

['CONTROL DE LLAMADAS', 'Hoja1', 'BASE', 'Hoja2', 'Hoja4']

## 1. Estructura del archivo

El archivo tiene 5 hojas:

- **CONTROL DE LLAMADAS**: bitácora de la encuesta telefónica (una fila por intento de llamada a una distribuidora). Tiene un encabezado "de formulario" a 4 filas (títulos de sección P1–P5 + subpreguntas condicionales), no un encabezado plano.
- **BASE**: padrón de distribuidoras (región, sucursal, coordinador, tipo/nivel, antigüedad, % disponible, etc.).
- **Hoja2**: información de crédito/saldo por distribuidora (límite, disponible, saldo, bolsa).
- **Hoja4**: listas de valores válidos usadas como listas desplegables en Excel (estatus de llamada, Sí/No) — no son datos, son catálogos.
- **Hoja1**: vacía.

La llave para relacionar las tres tablas de datos es `id_cliente` (en CONTROL DE LLAMADAS aparece como "Numero Dv").

In [2]:
raw = pd.read_excel(RUTA_EXCEL, sheet_name="CONTROL DE LLAMADAS", header=None)
raw.head(6)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,Nombre de Encuestador:,NaN,JUAN FELIPE RAMIREZ LARRIVA,NaN,NaN,NaN,NaN,P1 – Motivo principal,P2 – Calificación,Lógica condicional (IMPORTANTE),NaN,P3 – Competencia,NaN,P4 – Reactivación,P5 – Disposición Final,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pregunta:,Pregunta:,Si responde:,NaN,Pregunta:,Si responde “Sí” → activar pregunta:,Pregunta:,NaN,NaN,NaN
2,Fila,Fecha de llamada,Estatus de CONTACT,Plaza,Numero Dv,Nombre Dv,fhUltimoCanje,"“En una palabra o frase corta, ¿cuál es la pri...","“Del 1 al 5 donde 5 es muy buena y 1 es mala, ...","1, 2 o 3 → ir a sección “Área de Mejora” como ...",4 o 5 → ir a sección “Fortaleza”,“¿Actualmente trabajas con otra financiera?”,“¿Qué es lo que más te gusta de trabajar con e...,¿Qué Necesitas para ser exclusiva de CRECE CON...,“Si logramos resolver el punto que nos comenta...,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,“¿Qué podríamos mejorar?”,“¿Qué es lo que más valoras del servicio?”,NaN,NaN,NaN,Si,Tal vez,No
4,1,2026-02-20 00:00:00,No contestó,ACAYUCAN,86079,MARIA DE LOS ANGELES MEZA REYES,2026-01-30 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2,2026-02-20 00:00:00,No contestó,ACAYUCAN,86166,GABRIELA SANCHEZ GONZALEZ,2026-01-29 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Carga y limpieza de la encuesta (CONTROL DE LLAMADAS)

Se descartan las primeras 4 filas (encabezados de formulario) y se asignan nombres de columna explícitos según el mapeo de preguntas:

- P1 — Motivo principal (texto abierto): por qué no está colocando.
- P2 — Calificación (1–5) + condicional: si 1-3 → área de mejora; si 4-5 → fortaleza.
- P3 — Competencia: ¿trabaja con otra financiera? (Sí/No) + qué le gusta de ellos.
- P4 — Reactivación: qué necesita para ser exclusiva de CRECE CON VALES.
- P5 — Disposición final a ser recontactada (Sí / Tal vez / No), codificada en 3 columnas one-hot.

In [3]:
COLUMNAS = [
    "fila", "fecha_llamada", "estatus_contacto", "plaza", "id_cliente", "nombre_dv", "fh_ultimo_canje",
    "p1_motivo", "p2_calificacion", "p2_area_mejora", "p2_fortaleza",
    "p3_competencia", "p3_gusta_competencia", "p4_necesita_exclusiva",
    "p5_si", "p5_talvez", "p5_no",
]

encuesta = pd.read_excel(RUTA_EXCEL, sheet_name="CONTROL DE LLAMADAS", header=None, skiprows=4)
encuesta.columns = COLUMNAS

# Colapsar P5 (one-hot) en una sola columna categórica
def _p5(row):
    if row["p5_si"] == 1:
        return "Si"
    if pd.notna(row["p5_talvez"]):
        return "Tal vez"
    if pd.notna(row["p5_no"]):
        return "No"
    return pd.NA

encuesta["p5_disposicion"] = encuesta.apply(_p5, axis=1)
encuesta = encuesta.drop(columns=["p5_si", "p5_talvez", "p5_no"])

print(encuesta.shape)
encuesta.head()

(377, 15)


,fila,fecha_llamada,estatus_contacto,plaza,id_cliente,nombre_dv,fh_ultimo_canje,p1_motivo,p2_calificacion,p2_area_mejora,p2_fortaleza,p3_competencia,p3_gusta_competencia,p4_necesita_exclusiva,p5_disposicion
0,1.0,2026-02-20,No contestó,ACAYUCAN,86079.0,MARIA DE LOS ANGELES MEZA REYES,2026-01-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
1,2.0,2026-02-20,No contestó,ACAYUCAN,86166.0,GABRIELA SANCHEZ GONZALEZ,2026-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
2,3.0,2026-02-20,No contestó,ACAYUCAN,86052.0,MARIA DEL CARMEN CRUZ JIMENEZ,2025-12-23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
3,4.0,2026-02-20,No contestó,ACAYUCAN,109938.0,"PEREZ HERNANDEZ , PETRONILA",2026-01-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
4,5.0,2026-02-20,Llamada efectiva (contestó),ACAYUCAN,86237.0,HERLINDA LOPEZ NO,2026-02-01,"falta de apoyo del promotor, no se le da segui...",5.0,NaN,facilidades en el servicio de pagos,NO,NaN,NaN,<NA>


## 3. Calidad de los registros: ¿la llamada se contestó?

Este es el hallazgo central del diagnóstico: la mayoría de los intentos de llamada **no llegaron a generar una respuesta de encuesta**. Sólo las filas con `estatus_contacto == "Llamada efectiva (contestó)"` tienen datos de encuesta utilizables.

In [4]:
conteo_estatus = encuesta["estatus_contacto"].value_counts(dropna=False)
porcentaje_estatus = (conteo_estatus / len(encuesta) * 100).round(1)
pd.DataFrame({"n": conteo_estatus, "%": porcentaje_estatus})

,n,%
estatus_contacto,,
Buzón,142,37.7
No contestó,140,37.1
Llamada efectiva (contestó),64,17.0
Se negó a contestar,16,4.2
Reagendar llamada,11,2.9
NaN,3,0.8
Número cambiado,1,0.3


In [5]:
efectivas = encuesta[encuesta["estatus_contacto"] == "Llamada efectiva (contestó)"].copy()
print(f"Llamadas efectivas: {len(efectivas)} de {len(encuesta)} intentos ({len(efectivas)/len(encuesta)*100:.1f}%)")

# % de nulos dentro de las llamadas SI efectivas (aquí sí esperamos datos)
cols_encuesta = ["p1_motivo", "p2_calificacion", "p3_competencia", "p4_necesita_exclusiva", "p5_disposicion"]
(efectivas[cols_encuesta].isna().mean() * 100).round(1).rename("% nulos en llamadas efectivas")

Llamadas efectivas: 64 de 377 intentos (17.0%)


p1_motivo                 1.6
p2_calificacion           1.6
p3_competencia            4.7
p4_necesita_exclusiva    14.1
p5_disposicion           43.8
Name: % nulos en llamadas efectivas, dtype: float64

**Nota:** `p4_necesita_exclusiva` y `p5_disposicion` son condicionales (sólo aplican si la distribuidora dijo trabajar con la competencia / si se resolvió su queja), así que un % de nulos más alto ahí es esperado por diseño del cuestionario, no un problema de calidad.

## 4. Duplicados y llaves

In [6]:
dup = encuesta[encuesta.duplicated(subset="id_cliente", keep=False)].sort_values("id_cliente")
print(f"id_cliente duplicados: {encuesta['id_cliente'].duplicated().sum()}")
dup[["fila", "id_cliente", "nombre_dv", "estatus_contacto", "fecha_llamada"]]

id_cliente duplicados: 7


,fila,id_cliente,nombre_dv,estatus_contacto,fecha_llamada
215,528.0,104739.0,DE LOERA DUEÑAS MARTIN,No contestó,2026-02-21
218,528.0,104739.0,DE LOERA DUEÑAS MARTIN,Buzón,2026-02-21
217,527.0,105180.0,ROSALES ROSALES FERNANDO,Buzón,2026-02-21
214,527.0,105180.0,ROSALES ROSALES FERNANDO,No contestó,2026-02-21
216,526.0,108179.0,"MONTELONGO DE LA CRUZ , FABIOLA",No contestó,2026-02-21
213,526.0,108179.0,"MONTELONGO DE LA CRUZ , FABIOLA",Buzón,2026-02-21
170,482.0,110799.0,"DELGADO MORALES , MA GUADALUPE",No contestó,2026-02-21
168,482.0,110799.0,"DELGADO MORALES , MA GUADALUPE",No contestó,2026-02-21
167,481.0,112899.0,"BOCANEGRA TORRES , MA GUADALUPE",No contestó,2026-02-21
169,481.0,112899.0,"BOCANEGRA TORRES , MA GUADALUPE",No contestó,2026-02-21


## 5. Cruce con BASE (padrón) y Hoja2 (crédito/saldo)

In [7]:
base = pd.read_excel(RUTA_EXCEL, sheet_name="BASE")
hoja2 = pd.read_excel(RUTA_EXCEL, sheet_name="Hoja2")

print(f"encuesta: {encuesta['id_cliente'].nunique()} id_cliente únicos")
print(f"BASE:     {base['id_cliente'].nunique()} id_cliente únicos")
print(f"Hoja2:    {hoja2['id_cliente'].nunique()} id_cliente únicos")
print()
print(f"% ids de encuesta encontrados en BASE:  {encuesta['id_cliente'].isin(base['id_cliente']).mean()*100:.1f}%")
print(f"% ids de encuesta encontrados en Hoja2: {encuesta['id_cliente'].isin(hoja2['id_cliente']).mean()*100:.1f}%")

encuesta: 369 id_cliente únicos
BASE:     1915 id_cliente únicos
Hoja2:    2158 id_cliente únicos

% ids de encuesta encontrados en BASE:  99.2%
% ids de encuesta encontrados en Hoja2: 99.2%


In [8]:
base_cols = ["id_cliente", "region", "TIPO", "tipo.antiguedad", "%disponible", "tipo_colocado"]
hoja2_cols = ["id_cliente", "limite", "DISPONIBLE", "SALDO", "Bolsa", "Disponible_Bolsa"]

dataset = (
    encuesta
    .merge(base[base_cols].drop_duplicates("id_cliente"), on="id_cliente", how="left")
    .merge(hoja2[hoja2_cols].drop_duplicates("id_cliente"), on="id_cliente", how="left")
)
dataset.shape

(377, 25)

## 6. Criterio de inclusión propuesto

Para el análisis de contenido de la encuesta (P1–P5), se propone **incluir únicamente los registros con `estatus_contacto == "Llamada efectiva (contestó)"`**, ya que son los únicos con respuestas reales de la distribuidora. El resto (Buzón, No contestó, Se negó a contestar, Reagendar, Número cambiado) se documenta como parte de la tasa de contacto/respuesta, pero se excluye del análisis de contenido por no tener información que analizar.

El dataset completo (`dataset`, con variables de BASE/Hoja2) sí se conserva para todas las filas, por si se quiere caracterizar diferencias entre quién contesta y quién no (p. ej. por región, tipo, antigüedad, % disponible).

In [9]:
dataset_encuestados = dataset[dataset["estatus_contacto"] == "Llamada efectiva (contestó)"].copy()

import os
os.makedirs("../data/processed", exist_ok=True)
dataset.to_csv("../data/processed/dataset_completo.csv", index=False)
dataset_encuestados.to_csv("../data/processed/dataset_encuestados.csv", index=False)
print(f"Guardado: dataset_completo.csv ({len(dataset)} filas), dataset_encuestados.csv ({len(dataset_encuestados)} filas)")

Guardado: dataset_completo.csv (377 filas), dataset_encuestados.csv (64 filas)


## 7. Resumen del diagnóstico

- 377 intentos de llamada registrados; sólo **64 (17%)** resultaron en llamada efectiva con encuesta contestada.
- El resto del "nulo" en las columnas de preguntas es esperado (llamada no contestada), no un problema de captura.
- Dentro de las 64 llamadas efectivas, la tasa de respuesta a P1/P2/P3 es alta (>95%); P4/P5 son condicionales por diseño.
- 7 `id_cliente` duplicados en la bitácora de llamadas (revisar si son reintentos de contacto al mismo cliente).
- El cruce con BASE y Hoja2 es prácticamente perfecto (~100% de match), lo que permite enriquecer cada respuesta con región, tipo/nivel de distribuidora, antigüedad y datos de crédito.
- **Implicación para las siguientes etapas:** el análisis de contenido (texto, clustering, sentimiento) sólo tiene una base de **n=64** respuestas. Es una muestra chica para clustering robusto o NLP con muchos temas; hay que ser conservadores en el número de segmentos/temas y complementar con lectura cualitativa directa de las respuestas abiertas.